In [ ]:
import importlib
import numpy as np
import matplotlib.pyplot as plt

# adjust path
path_to_hkpt = '../'
import sys
sys.path.append(path_to_hkpt)

import hk_parallel_transport as hkpt
hkpt = importlib.reload(hkpt)

plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
})

cone_parallel_transport_explicit = hkpt.cone_parallel_transport_explicit

In [ ]:
def cone_metric_tensor(x, r):
    g = np.zeros((2, 2))
    g[0, 0] = r**2
    g[1, 1] = 1
    return g


def cone_geodesic_step(x0, r0, x1, r1, s):
    if s <= 0.0:
        return float(x0), float(r0)
    if s >= 1.0:
        return float(x1), float(r1)
    if r0 <= 1e-12 and r1 <= 1e-12:
        return float(x0), 0.0
    if r0 <= 1e-12:
        return float(x1), float(s * r1)
    if r1 <= 1e-12:
        return float(x0), float((1.0 - s) * r0)

    theta = min(abs(x1 - x0), np.pi)
    radius_sq = (
        (1.0 - s) ** 2 * r0**2
        + s**2 * r1**2
        + 2.0 * s * (1.0 - s) * r0 * r1 * np.cos(theta)
    )
    radius = np.sqrt(max(radius_sq, 0.0))
    if theta <= 1e-12 or radius <= 1e-12:
        return float(x0), float(radius)

    cos_arg = ((1.0 - s) * r0 + s * r1 * np.cos(theta)) / radius
    cos_arg = np.clip(cos_arg, -1.0, 1.0)
    rho = np.arccos(cos_arg) / theta
    position = (1.0 - rho) * x0 + rho * x1
    return float(position), float(radius)


def cone_tangent_to_cartesian(x, r, a, b):
    hor = b * np.cos(x) - r * a * np.sin(x)
    ver = b * np.sin(x) + r * a * np.cos(x)
    return np.array([hor, ver])

In [ ]:
# cone parallel transport visualization
initial_pos = np.array([1.0, 0.5]) # (x,r)
initial_vec = np.array([0.0, 0.75]) # (dx,dr)

final_pos = np.array([3.0, 1.0]) # (x,r)


# plot the cone tangent
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
ax_polar, ax_coords = axes
# convert to cartesian coordinates for plotting
pos_hor = initial_pos[1] * np.cos(initial_pos[0])
pos_ver = initial_pos[1] * np.sin(initial_pos[0])
pos_hor_final = final_pos[1] * np.cos(final_pos[0])
pos_ver_final = final_pos[1] * np.sin(final_pos[0])

cartesian_vec = cone_tangent_to_cartesian(initial_pos[0], initial_pos[1], initial_vec[0], initial_vec[1])
vec_hor = cartesian_vec[0]
vec_ver = cartesian_vec[1]
ax_polar.quiver(pos_hor, pos_ver, vec_hor, vec_ver, angles='xy', scale_units='xy', scale=1, color='black')
ax_polar.scatter(pos_hor, pos_ver, color='black', label='Initial Position')
ax_polar.scatter(pos_hor_final, pos_ver_final, color='black', label='Final Position')
ax_polar.set_xlim(-2, 2)
ax_polar.set_ylim(-2, 2)
# get rid of axes for better visualization
ax_polar.axis('off')
ax_polar.set_title(r'Polar coordinates, $\theta = x$', fontsize=20)
# draw circular grid lines for the cone
for r in np.linspace(0.5, 1.5, 3):
    circle_x = r * np.cos(np.linspace(0, 2*np.pi, 100))
    circle_y = r * np.sin(np.linspace(0, 2*np.pi, 100))
    ax_polar.plot(circle_x, circle_y, 'k--', alpha=0.5)


# draw geodesic from initial to final position
positions_hor_geodesic = []
positions_ver_geodesic = []
positions_x_geodesic = []
positions_r_geodesic = []
for s in np.linspace(0, 1, 40):
    position, radius = cone_geodesic_step(x0 = initial_pos[0], r0 = initial_pos[1], x1 = final_pos[0], r1 = final_pos[1], s = s)
    pos_hor_geodesic = radius * np.cos(position)
    pos_ver_geodesic = radius * np.sin(position)
    positions_hor_geodesic.append(pos_hor_geodesic)
    positions_ver_geodesic.append(pos_ver_geodesic)
    positions_x_geodesic.append(position)
    positions_r_geodesic.append(radius)
ax_polar.plot(positions_hor_geodesic, positions_ver_geodesic, color='black', label='Geodesic Path')
# parallel transported vector
transported_vec = cone_parallel_transport_explicit(a0 = initial_vec[0], b0 = initial_vec[1], z0 = initial_pos, z1 = final_pos)
# convert to cartesian coordinates for plotting
transported_cartesian_vec = cone_tangent_to_cartesian(final_pos[0], final_pos[1], transported_vec[0], transported_vec[1])
vec_hor_final = transported_cartesian_vec[0]
vec_ver_final = transported_cartesian_vec[1]
ax_polar.quiver(pos_hor_final, pos_ver_final, vec_hor_final, vec_ver_final, angles='xy', scale_units='xy', scale=1, color='black', label='Parallel Transported Vector')

ax_coords.plot(positions_x_geodesic, positions_r_geodesic, color='black', label='Geodesic Path')
ax_coords.scatter(initial_pos[0], initial_pos[1], color='black', label='Initial Position')
ax_coords.scatter(final_pos[0], final_pos[1], color='black', label='Final Position')
ax_coords.quiver(initial_pos[0], initial_pos[1], initial_vec[0], initial_vec[1], angles='xy', scale_units='xy', scale=1, color='black')
ax_coords.quiver(final_pos[0], final_pos[1], transported_vec[0], transported_vec[1], angles='xy', scale_units='xy', scale=1, color='black', label='Parallel Transported Vector')
x_pad = 0.3
r_pad = 0.3
ax_coords.set_xlim(min(initial_pos[0], final_pos[0]) - x_pad, max(initial_pos[0], final_pos[0]) + x_pad)
ax_coords.set_ylim(0.0, max(initial_pos[1], final_pos[1]) + r_pad)
ax_coords.set_xlabel(r'$x$', fontsize=20)
ax_coords.set_ylabel(r'$r$', fontsize=20)
ax_coords.set_title('Cone coordinates $(x, r)$', fontsize=20)
plt.tight_layout()
plt.savefig('../Figures/cone_parallel_transport_visualization.pdf', dpi=800)
